In [1]:
import glob
import math
import numpy as np
import ROOT
from utils.hist_utils import get_highest_channels, fit_and_scale_mc

ROOT.EnableImplicitMT()
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/Include/Codes/")

ROOT.gInterpreter.Declare('#include "const.h"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

In [37]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files_original = glob.glob("/workspace/root_files/cnaf/2026-09-06/Signal/ALL_PHYS_SIGNAL_NoSmearing/*.root")
all_files_original += glob.glob("/workspace/root_files/cnaf/2026-09-06/Signal/ALL_PHYS2_SIGNAL_NoSmearing/*.root")
all_files_original += glob.glob("/workspace/root_files/cnaf/2026-09-06/Signal/ALL_PHYS3_SIGNAL_NoSmearing/*.root")

all_files = glob.glob("/workspace/root_files/cnaf/*.root")

def get_valid_files(file_list):
    valid = []
    for f in file_list:
        try:
            tfile = ROOT.TFile.Open(f, "READ")
            if tfile and not tfile.IsZombie() and tfile.GetNkeys() > 0:
                valid.append(f)
            else:
                print(f"Pomijam uszkodzony/pusty plik: {f}")
            if tfile:
                tfile.Close()
        except Exception:
            print(f"Błąd otwarcia pliku (pomijam): {f}")
    return valid

all_files = get_valid_files(all_files)
all_files_original = get_valid_files(all_files_original)

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df = ROOT.RDataFrame("h1", all_files).Filter("EclFilfo & 2")
df_original = ROOT.RDataFrame("h1", all_files_original).Filter("mctruth == 1 || mctruth == 0 || mctruth == -1").Filter("Eclfilfo & 2")

## Runs in the files

In [41]:
ROOT.gStyle.SetOptStat(0)
canva = ROOT.TCanvas("canva", "", 800, 800)

hist_mc = df.Histo1D(("nrun_mc", ";Run Number;Counts", 100, 30300, 41902), "nRun")

hist_mc_original = df_original.Histo1D(("nrun_mc_orig", ";Run Number;Counts", 100, 30300, 41902), "nrun")
hist_mc_original.Scale(1000)

leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
leg.SetBorderSize(0)
leg.SetFillStyle(0)

hist_mc.SetLineColor(channColor.get("MC sum", ROOT.kOrange))
hist_mc_original.SetLineColor(ROOT.kBlack)

leg.AddEntry(hist_mc.GetPtr(), "MC Sum", "l")

hist_mc_original.Draw("HIST")
hist_mc.Draw("HIST SAME")
leg.Draw("SAME")

canva.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


In [48]:
hist_list = []
hist_list.append((df.Histo1D(("h_rho_ch", ";#rho_{ch} [cm];Counts", 201, 0, 12), "nClu")))
# hist_list.append((df.Histo1D(("h_z_ch", ";z_{ch} - z_{IP} [cm];Counts", 201, 0, 6), "nT")))
hist_list.append((df_original.Histo1D(("h_nclu_original", ";#rho_{ch} [cm];Counts", 201, 0, 12), "nclu")))
# hist_list.append((df_original.Histo1D(("h_nt_original", ";z_{ch} - z_{IP} [cm];Counts", 201, 0, 6), "nt")))
        
ROOT.RDF.RunGraphs(hist_list)

2

In [51]:
ROOT.gStyle.SetOptStat(0)

canvadivision = ROOT.TCanvas("canva", "", 800, 800)

hist_list[0].SetLineColor(ROOT.kOrange)
hist_list[1].SetLineColor(ROOT.kBlack)
hist_list[1].Scale(20)

hist_list[0].Draw("HIST")
hist_list[1].Draw("HIST SAME")

canvadivision.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva
